# Evaluación final y priorización comercial
Lourdes Lema · UDLA · Bloque 5 · v01

Propósito: evaluar en septiembre el diseño cerrado mediante validación. RF09 queda fijado antes de abrir la prueba: 400 árboles, profundidad 8, mínimo 1 por hoja y 3 variables candidatas. Regresión lineal conserva su especificación. No se seleccionan parámetros ni se cambia el ganador de validación con septiembre.

Ejecutar las celdas en orden en Google Colab, guardando primero una copia en Drive. Subir el ZIP de cortes v06 y el ZIP de validación RF que ya se verificó. No descomprimirlos. Se descarga un ZIP de evidencias al terminar.

Las salidas incluidas, si existen, son de ejecución local. La reproducción en Colab debe conservar las versiones de su validación: scikit-learn 1.6.1. Si el entorno no coincide, detenerse y revisar antes de ejecutar; no instalar ni cambiar versiones silenciosamente.

## Carga y cierre de decisiones
**Propósito:** registrar las decisiones antes de consultar la prueba.
**Operaciones:** verificar huellas, leer la selección de validación y escribir protocolo previo.
**Resultado esperado:** RF09 y los cinco predictores acordados, sin cambios.
**Interpretación:** los resultados finales evaluarán estas decisiones, incluso si fueran desfavorables.


In [ ]:
from pathlib import Path
import io, json, zipfile, hashlib, platform, math
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
try:
    from IPython.display import display
except ImportError:
    def display(d): print(d.to_string(index=False))
RUTA_CORTES = None
RUTA_RF = None
PERMITIR_ENTORNO_LOCAL = False # Solo ejecución técnica documentada; en Colab conservar False.
def cargar(ruta, mensaje):
    if ruta is not None: return Path(ruta).read_bytes()
    from google.colab import files
    print(mensaje)
    recibido=files.upload()
    assert len(recibido)==1, 'Subir un archivo por solicitud'
    return next(iter(recibido.values()))
sha=lambda b:hashlib.sha256(b).hexdigest()
rfbytes=cargar(RUTA_RF,'Subir Evidencias_RF_validacion_v01.zip')
assert sha(rfbytes)=='d97601a291d3a7132485a6b83c73ec960a4752ccef93acd0c6c49fed4cc29d42', 'El ZIP RF no coincide con la evidencia verificada'
HUELLAS={'entrenamiento_inicial_v06.csv': '65d7dc09069779841830e0d596ecd4eb6cd78a61f193b12d8a7ffc08a356cd59', 'validacion_v06.csv': 'bfb661e6f35357af531beebfba7f6bf0acffeb126a0a259c129ad4479a8bcedc', 'prueba_reservada_v06.csv': '574a154321c7902019d8bb0ba09b8c686ba929ca9c1d376bd71d4f67d19adbec'}
with zipfile.ZipFile(io.BytesIO(rfbytes)) as z:
    seleccion=json.loads(z.read('seleccion_RF.json'))
    bitacora_validacion=json.loads(z.read('bitacora.json'))
    comparacion_validacion=pd.read_csv(io.BytesIO(z.read('comparacion_validacion.csv')))
assert seleccion['configuracion_RF']==dict(id='RF09',n_estimators=400,max_depth=8,min_samples_leaf=1,max_features=3)
versiones=dict(python=platform.python_version(),pandas=pd.__version__,numpy=np.__version__,sklearn=sklearn.__version__)
assert PERMITIR_ENTORNO_LOCAL or sklearn.__version__==bitacora_validacion['versiones']['scikit-learn'], 'Versión sklearn distinta: revisar entorno antes de continuar'
XCOL=['recencia_dias','frecuencia_180d','diversidad_180d','importe_neto_180d','importe_neto_por_compra_180d']
Y='objetivo_neto_90d'; H='referencia_neto_90d'
out=Path('salidas_prueba_final_v01');out.mkdir(exist_ok=True)
def guardar_json(nombre,obj): (out/nombre).write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf-8')
protocolo=dict(fecha_utc=datetime.now(timezone.utc).isoformat(),seleccion=seleccion['configuracion_RF'],estado='Validación cotejada; configuración cerrada antes de prueba',predictores=XCOL,criterio='MAE promedio marzo y junio con igual peso',capacidades=[0.1,0.2,0.3],desempate='Customer ID texto ascendente',tamano='floor(p*n)',transformaciones='Ninguna; sin recorte de predicciones',versiones=versiones,sha256_RF=sha(rfbytes))
guardar_json('protocolo_previo.json',protocolo)
cortesbytes=cargar(RUTA_CORTES,'Subir Evidencias_cortes_v06(1).zip')
with zipfile.ZipFile(io.BytesIO(cortesbytes)) as z:
    tablas=[]
    for n,h in HUELLAS.items():
        b=z.read(n);assert sha(b)==h, 'CSV distinto: '+n
        tablas.append(pd.read_csv(io.BytesIO(b),dtype={'Customer ID':'string'},parse_dates=['fecha_corte']))
inicial,validacion,prueba=tablas
train=pd.concat([inicial,validacion],ignore_index=True)


## Auditoría de la partición final
**Propósito:** asegurar que el reajuste representa información disponible al 1 de septiembre de 2011.
**Operaciones:** revisar claves, tipos, valores finitos, fórmula P=M/F, conteos y fin de etiquetas; ordenar filas de forma reproducible.
**Resultado esperado:** 14.935 filas de entrenamiento y 2.768 de prueba; última etiqueta de entrenamiento termina el 30 de agosto y la prueba termina el 30 de noviembre de 2011, dentro de la cobertura previamente auditada.
**Interpretación:** un cliente puede aparecer en ambos conjuntos. La separación es temporal, no por identidad. La elegibilidad y reconstrucción desde transacciones se comprobaron en el bloque de auditoría; aquí se comprueba su integridad por huellas y controles.


In [ ]:
controles=[]
def verificar(nombre,cond):
    controles.append(dict(control=nombre,aprobado=bool(cond)))
    assert cond,nombre
for nombre,d in [('train',train),('prueba',prueba)]:
    verificar(nombre+' claves únicas',not d.duplicated(['Customer ID','fecha_corte']).any())
    verificar(nombre+' sin nulos',not d.isna().any().any())
    verificar(nombre+' finitos',np.isfinite(d[XCOL+[Y,H]]).all().all())
    verificar(nombre+' frecuencia entera positiva',((d.frecuencia_180d>=1)&(d.frecuencia_180d%1==0)).all())
    verificar(nombre+' diversidad entera positiva',((d.diversidad_180d>=1)&(d.diversidad_180d%1==0)).all())
    verificar(nombre+' recencia histórica',d.recencia_dias.between(0,180,inclusive='right').all())
    verificar(nombre+' P=M/F',np.allclose(d.importe_neto_por_compra_180d,d.importe_neto_180d/d.frecuencia_180d,rtol=1e-12,atol=1e-10))
    d.sort_values(['fecha_corte','Customer ID'],inplace=True);d.reset_index(drop=True,inplace=True)
verificar('14935 entrenamiento',len(train)==14935)
verificar('2768 prueba',len(prueba)==2768)
verificar('conteos por corte',train.groupby('fecha_corte').size().tolist()==[2652,2822,3465,3338,2658])
verificar('fechas entrenamiento',set(train.fecha_corte.dt.strftime('%Y-%m-%d'))=={'2010-06-01','2010-09-01','2010-12-01','2011-03-01','2011-06-01'})
verificar('corte prueba',prueba.fecha_corte.eq(pd.Timestamp('2011-09-01')).all())
verificar('etiquetas disponibles',((train.fecha_corte+pd.Timedelta(days=90))<=pd.Timestamp('2011-09-01')).all())
verificar('cobertura completa prueba',pd.Timestamp('2011-09-01')+pd.Timedelta(days=90)<=pd.Timestamp('2011-12-09'))
print('Entrenamiento:',len(train),'Prueba:',len(prueba))


## Reajuste y evaluación predictiva
**Propósito:** medir generalización temporal en un período reservado.
**Operaciones:** entrenar desde cero RL y RF09 con las mismas cinco variables; predecir septiembre; calcular MAE, RMSE y R²; conservar predicciones negativas.
**Resultado esperado:** tres filas de métricas y una predicción por cliente para cada modelo, junto con H.
**Interpretación:** MAE expresa error absoluto promedio en GBP; RMSE da mayor peso a errores grandes; R² puede ser negativo. La comparación final no reemplaza el criterio de selección usado en validación. No se estiman efectos causales ni significancia estadística.


In [ ]:
rl=LinearRegression()
rf=RandomForestRegressor(**seleccion['parametros_efectivos'])
rl.fit(train[XCOL],train[Y]);rf.fit(train[XCOL],train[Y])
pred=prueba[['Customer ID','fecha_corte',Y,H]].copy()
pred['Regresión lineal']=rl.predict(prueba[XCOL]);pred['Random Forest RF09']=rf.predict(prueba[XCOL])
pred.rename(columns={H:'Referencia H'},inplace=True)
modelos=['Referencia H','Regresión lineal','Random Forest RF09']
y=pred[Y].to_numpy();metricas=[]
for nombre in modelos:
    v=pred[nombre].to_numpy();verificar(nombre+' predicciones finitas',np.isfinite(v).all())
    e=y-v;sct=((y-y.mean())**2).sum()
    metricas.append(dict(modelo=nombre,n=len(y),MAE_GBP=np.abs(e).mean(),RMSE_GBP=np.sqrt(np.mean(e**2)),R2=1-np.sum(e**2)/sct if sct>0 else None,predicciones_negativas=int((v<0).sum())))
metricas=pd.DataFrame(metricas)
display(metricas)
pd.DataFrame({'variable':['intercepto']+XCOL,'coeficiente':[rl.intercept_]+list(rl.coef_)}).to_csv(out/'coeficientes_RL.csv',index=False)
guardar_json('parametros_efectivos.json',{'RL':rl.get_params(),'RF09':rf.get_params()})


## Evaluación comercial bajo capacidad limitada
**Propósito:** comprobar si ordenar por predicción identifica grupos con mayores ventas futuras observadas que ordenar por H.
**Operaciones:** ordenar valores originales descendentes y desempatar por identificador textual ascendente; seleccionar floor(p×2768): 276, 553 y 830 clientes; sumar sus ventas futuras netas y comparar con H.
**Resultado esperado:** nueve filas de evaluación comercial y detalle de clientes seleccionados.
**Interpretación:** la diferencia en GBP compara ventas ya observadas de grupos de igual tamaño. No representa ventas causadas por contactar clientes. La proporción solo se interpreta si el total neto de la cohorte es positivo; puede ser negativa o superar 100 %. Los tres escenarios son capacidades hipotéticas. No se elige retrospectivamente el porcentaje más favorable.


In [ ]:
total=float(pred[Y].sum());filas=[];seleccionados=[]
for porcentaje in [10,20,30]:
    k=(porcentaje*len(pred))//100
    base=pred.sort_values(['Referencia H','Customer ID'],ascending=[False,True],kind='mergesort').head(k)
    suma_h=float(base[Y].sum())
    for nombre in modelos:
        grupo=pred.sort_values([nombre,'Customer ID'],ascending=[False,True],kind='mergesort').head(k).copy()
        suma=float(grupo[Y].sum())
        verificar(f'{nombre} {porcentaje}% tamaño',len(grupo)==k and grupo['Customer ID'].nunique()==k)
        filas.append(dict(modelo=nombre,capacidad_pct=porcentaje,n_seleccionados=k,ventas_futuras_netas_GBP=suma,ventas_seleccion_H_GBP=suma_h,diferencia_frente_H_GBP=suma-suma_h,total_cohorte_GBP=total,concentracion_pct=100*suma/total if total>0 else None,diferencia_concentracion_pp=100*(suma-suma_h)/total if total>0 else None,clientes_comunes_H=len(set(grupo['Customer ID'])&set(base['Customer ID']))))
        grupo['criterio']=nombre;grupo['capacidad_pct']=porcentaje;grupo['posicion']=np.arange(1,k+1);seleccionados.append(grupo)
comercial=pd.DataFrame(filas);detalle=pd.concat(seleccionados,ignore_index=True)
display(comercial)
print('Total neto cohorte GBP:',total)
print('Un objetivo cero no permite distinguir por sí solo ausencia de movimientos y compensaciones.')


## Evidencias y cierre del bloque
**Propósito:** conservar resultados verificables y reproducibles.
**Operaciones:** exportar métricas, predicciones sin redondeo deliberado, selección, controles, parámetros, versiones y huellas. No se modifica la evidencia original de validación.
**Resultado esperado:** Evidencias_prueba_final_v01.zip.
**Interpretación:** compartir este ZIP para cotejar resultados antes de redactar conclusiones. Si la prueba muestra resultados desfavorables, se informan; no se reutiliza para ajustar el diseño. La visualización y propuesta se desarrollarán después de esta revisión.


In [ ]:
for nombre,d in [('metricas_prueba.csv',metricas),('priorizacion_prueba.csv',comercial),('predicciones_prueba.csv',pred),('clientes_seleccionados.csv',detalle),('controles.csv',pd.DataFrame(controles)),('comparacion_validacion.csv',comparacion_validacion)]: d.to_csv(out/nombre,index=False)
nota='Evaluación retrospectiva final. Configuración RF09 fijada mediante validación; prueba no usada para ajuste. Resultados de priorización descriptivos, sin inferencia causal. Limitaciones: dos cortes de validación, una prueba, clientes repetidos, ventanas históricas solapadas, búsqueda acotada, una semilla y sin análisis de sensibilidad a repeticiones. La ejecución final se realizó en Google Colab. La bitácora registra el entorno y las huellas de las entradas y salidas cotejadas. La matriz cliente-corte reconstruida de forma independiente continúa pendiente de cotejo celda por celda con v06.'
(out/'nota_interpretacion.txt').write_text(nota,encoding='utf-8')
guardar_json('bitacora.json',dict(fecha_utc=datetime.now(timezone.utc).isoformat(),versiones=versiones,ejecucion_local=PERMITIR_ENTORNO_LOCAL,sha256_cortes=sha(cortesbytes),sha256_validacion_RF=sha(rfbytes),huellas_entradas=HUELLAS,controles_aprobados=len(controles),n_train=len(train),n_prueba=len(prueba),limite_etiqueta_train=str((train.fecha_corte+pd.Timedelta(days=90)).max()),fin_prueba='2011-11-30',sha256_salidas={p.name:sha(p.read_bytes()) for p in out.iterdir() if p.name!='bitacora.json'}))
zip_salida=Path('Evidencias_prueba_final_v01.zip')
with zipfile.ZipFile(zip_salida,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(out.iterdir()): z.write(p,p.name)
print('Controles aprobados:',len(controles),'Archivo:',zip_salida)
try:
    from google.colab import files
    files.download(str(zip_salida))
except ImportError:
    print('Ejecución local: archivo disponible en el directorio de trabajo.')
